# Phycobilisome Row-Based Particle List Cleaning

Cleans a RELION5 template-matching particle STAR file by assigning row identities to phycobilisome particles arranged in rows on thylakoid membranes.

**Pipeline overview:**
1. Load STAR file from template matching
2. Assign row identities via spatial + angular graph clustering
3. Fill row gaps via a dilation → reclassification → erosion strategy
4. Remove duplicate particles (e.g. from dilation)
5. Extract and save singleton particles separately
6. Assign halfsets by row identity (row-level random subset assignment)
7. Extract row endpoints; filter those near the lamella boundary
8. Export: cleaned STAR, singleton STAR, endpoint STAR, and QC PDF

**Dependencies:** numpy, pandas, scipy, scikit-learn, networkx, matplotlib, rtree, mrcfile, scikit-image, trimesh (optional)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import random

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits.mplot3d import Axes3D
import rtree

import networkx as nx
from collections import defaultdict
from itertools import combinations
from typing import Tuple, List, Dict
from dataclasses import dataclass

from scipy.interpolate import make_interp_spline
from scipy.signal     import find_peaks
from scipy.spatial    import cKDTree
from scipy.spatial.distance import pdist, cdist, squareform
from scipy.spatial.transform import Rotation as R
from scipy.stats      import chisquare, gaussian_kde, norm
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.mixture  import GaussianMixture
from sklearn.neighbors import NearestNeighbors

## 2. User Parameters

All parameters that may require adjustment for a new dataset are collected here.

In [ ]:
# ── Input / output paths ─────────────────────────────────────────────────────

INPUT_STAR       = 'particles.star'          # RELION5 STAR file from template matching
OUTPUT_STAR      = 'particles_rowID.star'    # Output: cleaned particles with RowID and RandomSubset
SINGLETONS_STAR  = 'particles_singletons.star'  # Output: singleton particles (no row assignment)
ENDPOINTS_STAR   = 'particles_endpoints.star'   # Output: row endpoint particles
QC_PDF           = 'rows_by_tomogram.pdf'    # Output: per-tomogram QC plot

# ── Tomogram metadata ────────────────────────────────────────────────────────

BMASK_DIR        = '../bmasks'               # Directory containing binary lamella masks as <tomo_num>.mrc
TOMOLIST_PATH    = '../tomolist.txt'         # Two-column text file: tomo_number  tomo_name (one per line)
                                             # Alternatively, define a dict `tomo_mapping = {num: name, ...}`
                                             # in a cell above and it will be used automatically.

# ── Row clustering parameters ────────────────────────────────────────────────

SPATIAL_TH       = 160   # Å — maximum inter-particle distance to form a row edge.
                         # Should reflect the expected nearest-neighbour distance within a row.
ANGLE_TH_ROT     = 20    # degrees — maximum |ΔRot|  between neighbours to form an edge
ANGLE_TH_TILT    = 10    # degrees — maximum |ΔTilt| between neighbours
ANGLE_TH_PSI     = 20    # degrees — maximum |ΔPsi|  between neighbours
K_NEIGHBORS      = 5     # Number of nearest neighbours to evaluate per particle

# ── Duplicate removal ────────────────────────────────────────────────────────

DUP_TH           = 120   # Å — particles closer than this after dilation/erosion are deduplicated.
                         # Original (non-ghost) particles are preferred; ties broken by LCC score.

# ── Edge exclusion ───────────────────────────────────────────────────────────

EDGE_EXCLUDE_A   = 200.0  # Å — row endpoints closer than this to the lamella boundary are removed.

## 3. Helper Functions

Run once. Do not modify unless adapting the pipeline to a different data format.

In [ ]:
# ── STAR file I/O ─────────────────────────────────────────────────────────────

def read_data_particles_star(path):
    """Parse a RELION5 STAR file into a DataFrame, preserving header for re-writing.
    
    Returns
    -------
    df         : pd.DataFrame of particle data
    lines      : raw file lines (for header reconstruction)
    dp_idx     : line index of 'data_particles'
    loop_idx   : line index of 'loop_'
    data_start : line index of first data row
    cols       : ordered list of column names
    """
    with open(path) as f:
        lines = f.readlines()
    dp_idx = next(i for i, L in enumerate(lines) if L.strip().lower() == 'data_particles')
    loop_idx = next(j for j in range(dp_idx+1, len(lines))
                    if lines[j].strip().lower() == 'loop_')
    cols = []
    for k in range(loop_idx+1, len(lines)):
        txt = lines[k].strip()
        if txt.startswith('_rln'):
            name = txt.split()[0]
            if name not in cols:
                cols.append(name)
            continue
        if txt and (txt[0].isdigit() or txt[0] in '+-'):
            data_start = k
            break
    data = []
    for row in lines[data_start:]:
        row = row.strip()
        if not row or row.startswith('#') or row.startswith('_rln'):
            continue
        parts = row.split(None, len(cols)-1)
        if len(parts) != len(cols):
            continue
        data.append(parts)
    df = pd.DataFrame(data, columns=cols)
    return df, lines, dp_idx, loop_idx, data_start, cols


def write_augmented_star(df, lines, dp_idx, loop_idx, data_start, orig_cols, output_path):
    """Write a STAR file containing the original columns plus _rlnRowID and _rlnRandomSubset."""
    n_orig = len(orig_cols)
    col_lines = lines[loop_idx+1:data_start]
    new_cols = [
        f"_rlnRowID            #{n_orig+1}\n",
        f"_rlnRandomSubset     #{n_orig+2}\n"
    ]
    header = lines[:loop_idx+1] + col_lines + new_cols
    with open(output_path, 'w') as out:
        out.writelines(header)
        for _, row in df.iterrows():
            values = [row[c] for c in orig_cols]
            values.append(str(int(row['RowID'])))
            values.append(row['_rlnRandomSubset'])
            out.write('  '.join(map(str, values)) + '\n')

In [ ]:
# ── Row assignment ────────────────────────────────────────────────────────────

def passes_angle_threshold(e1, e2):
    """Return True if all three Euler angle differences are within configured thresholds."""
    return (abs(e1[0] - e2[0]) <= ANGLE_TH_ROT and
            abs(e1[1] - e2[1]) <= ANGLE_TH_TILT and
            abs(e1[2] - e2[2]) <= ANGLE_TH_PSI)


def assign_halfsets_from_rowid(df, rowid_col='RowID'):
    """Assign half-sets (1 or 2) at row level, balancing total particle count across subsets.
    
    Entire rows are assigned to a single halfset to avoid correlated noise between halves.
    """
    sizes = df[rowid_col].value_counts().sort_index()
    row_ids = sizes.index.tolist()
    np.random.shuffle(row_ids)
    sum1 = sum2 = 0
    assignment = {}
    for rid in row_ids:
        if sum1 <= sum2:
            assignment[rid] = '1'
            sum1 += sizes[rid]
        else:
            assignment[rid] = '2'
            sum2 += sizes[rid]
    df['_rlnRandomSubset'] = df[rowid_col].map(assignment)
    return df

In [ ]:
# ── Dilation / erosion (gap closing) ─────────────────────────────────────────

def _principal_axis(points: np.ndarray) -> np.ndarray:
    """Return the principal axis (eigenvector of max eigenvalue) of a point cloud."""
    if points.shape[0] < 2:
        return np.array([1.0, 0.0, 0.0], float)
    X = points - points.mean(axis=0, keepdims=True)
    C = (X.T @ X) / max(1, X.shape[0] - 1)
    w, v = np.linalg.eigh(C)
    axis = v[:, np.argmax(w)]
    return axis / (np.linalg.norm(axis) + 1e-12)

def _order_along_axis(points: np.ndarray, axis: np.ndarray):
    t = points @ axis
    return np.argsort(t), t

def _median_step(points: np.ndarray, order_idx: np.ndarray) -> float:
    if len(order_idx) < 2:
        return 0.0
    diffs = np.linalg.norm(np.diff(points[order_idx], axis=0), axis=1)
    return float(np.median(diffs)) if diffs.size else 0.0


def _dilate_add_ghosts_using_existing_RowID(df_in: pd.DataFrame) -> pd.DataFrame:
    """Extend each existing row by one ghost particle at each endpoint.
    
    Ghost positions are extrapolated from the terminal inter-particle step vector.
    Ghost particles carry identical angles to their originating endpoint.
    """
    X, Y, Z = '_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst'
    TOMO = '_rlnTomoName'
    dfw = df_in.copy()
    if 'rlnIsGhost' not in dfw.columns:
        dfw['rlnIsGhost'] = 0
    out_blocks, ghost_blocks = [], []
    for tomo, block in dfw.groupby(TOMO, sort=False):
        out_blocks.append(block)
        for rid, sub in block.groupby('RowID', sort=False):
            if (rid < 0) or (len(sub) < 2):
                continue
            pts = sub[[X, Y, Z]].to_numpy(float)
            axis = _principal_axis(pts)
            order_idx, _ = _order_along_axis(pts, axis)
            step = _median_step(pts, order_idx)
            if step <= 0:
                continue
            i0, i1 = order_idx[0], order_idx[-1]
            p0, p1 = pts[i0], pts[i1]
            v0 = (p0 - pts[order_idx[1]]) if len(order_idx) >= 2 else axis
            v1 = (p1 - pts[order_idx[-2]]) if len(order_idx) >= 2 else axis
            v0 = v0 / (np.linalg.norm(v0) + 1e-12)
            v1 = v1 / (np.linalg.norm(v1) + 1e-12)
            g0 = sub.iloc[i0].copy(); g1 = sub.iloc[i1].copy()
            g0[[X, Y, Z]] = p0 + step * v0
            g1[[X, Y, Z]] = p1 + step * v1
            g0['rlnIsGhost'] = 1; g1['rlnIsGhost'] = 1
            ghost_blocks.append(pd.DataFrame([g0, g1], columns=dfw.columns))
    df_aug = pd.concat(out_blocks + ghost_blocks, ignore_index=True) if ghost_blocks \
             else pd.concat(out_blocks, ignore_index=True)
    return df_aug


def _reclassify_rows(df_in: pd.DataFrame) -> pd.DataFrame:
    """Re-run the spatial + angular graph clustering to assign RowID from scratch."""
    df_out = df_in.copy()
    df_out['RowID'] = -1
    global_id = 0
    for tomo in df_out['_rlnTomoName'].unique():
        sub    = df_out[df_out['_rlnTomoName'] == tomo]
        coords = sub[['_rlnCenteredCoordinateXAngst',
                      '_rlnCenteredCoordinateYAngst',
                      '_rlnCenteredCoordinateZAngst']].to_numpy()
        eulers = sub[['_rlnAngleRot', '_rlnAngleTilt', '_rlnAnglePsi']].to_numpy()
        idxs   = sub.index.to_numpy()
        nbrs     = NearestNeighbors(n_neighbors=K_NEIGHBORS).fit(coords)
        _, neigh = nbrs.kneighbors(coords)
        G = nx.Graph()
        G.add_nodes_from(idxs)
        for i, neighbors in enumerate(neigh):
            for j in neighbors[1:]:
                if np.linalg.norm(coords[i] - coords[j]) > SPATIAL_TH:
                    continue
                if passes_angle_threshold(eulers[i], eulers[j]):
                    G.add_edge(idxs[i], idxs[j])
        for comp in nx.connected_components(G):
            if len(comp) > 1:
                for ii in comp:
                    df_out.at[ii, 'RowID'] = global_id
                global_id += 1
    return df_out


def _erode_drop_ghost_ends(df_in: pd.DataFrame) -> pd.DataFrame:
    """Remove ghost particles that remain as MST leaf nodes after reclassification."""
    X, Y, Z = '_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst'
    kept = []
    for (tomo, rid), sub in df_in.groupby(['_rlnTomoName', 'RowID'], sort=False):
        if rid < 0 or len(sub) <= 1:
            kept.append(sub); continue
        coords = sub[[X, Y, Z]].to_numpy(float)
        D   = squareform(pdist(coords, metric='euclidean'))
        csr = minimum_spanning_tree(D)
        A   = (csr + csr.T).astype(bool).astype(int)
        deg = np.ravel(A.sum(axis=1))
        endpoints = np.where(deg == 1)[0]
        to_drop = [sub.index[iloc] for iloc in endpoints
                   if int(sub.iloc[iloc].get('rlnIsGhost', 0)) == 1]
        kept.append(sub.loc[~sub.index.isin(to_drop)].copy() if to_drop else sub)
    return pd.concat(kept, ignore_index=True)


def close_row_gaps(df: pd.DataFrame):
    """One round of dilation → reclassification → erosion to merge adjacent row fragments.
    
    Requires df to already contain a 'RowID' column from an initial classification pass.
    Returns (closed_df, per_tomo_stats_df).
    """
    if 'RowID' not in df.columns:
        raise KeyError("df must already contain 'RowID' before gap closing.")
    df_aug    = _dilate_add_ghosts_using_existing_RowID(df)
    df_re     = _reclassify_rows(df_aug)
    df_closed = _erode_drop_ghost_ends(df_re)
    rows_before = int(df['RowID'].max() + 1) if (df['RowID'] >= 0).any() else 0
    rows_after  = int(df_re['RowID'].max() + 1) if (df_re['RowID'] >= 0).any() else 0
    per_tomo = []
    for tomo, sub in df.groupby('_rlnTomoName'):
        rb = int(sub['RowID'].max() + 1) if (sub['RowID'] >= 0).any() else 0
        ra = int(df_closed.loc[df_closed['_rlnTomoName'] == tomo, 'RowID'].max() + 1) \
             if (df_closed.loc[df_closed['_rlnTomoName'] == tomo, 'RowID'] >= 0).any() else 0
        per_tomo.append(dict(tomo=tomo, rows_before=rb, rows_after=ra))
    return df_closed, pd.DataFrame(per_tomo)

In [ ]:
# ── Duplicate removal ─────────────────────────────────────────────────────────

def _first_available_score(sub, score_cols):
    """Return a float score vector using the first available column; NaNs → -inf."""
    for col in score_cols:
        if col in sub.columns:
            v = pd.to_numeric(sub[col], errors='coerce').to_numpy(float)
            v[~np.isfinite(v)] = -np.inf
            return v
    return np.zeros(len(sub), dtype=float)


def clean_with_kdtree_prefer_real(
    df: pd.DataFrame,
    min_dist: float,
    tomo_column: str = '_rlnTomoName',
    score_cols = ('_rlnLCCmax', '_rlnAutopickFigureOfMerit', '_rlnLogLikeliContribution',
                  '_rlnMaxValueProbDistribution', '_rlnPeakCC', '_rlnSNR'),
    ghost_col: str = 'rlnIsGhost',
):
    """Remove duplicate particles closer than min_dist within each tomogram.
    
    Priority: non-ghost > ghost; higher score wins ties.
    Returns (kept_df, removed_df).
    """
    X, Y, Z = '_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst'
    work = df.copy()
    for c in (X, Y, Z):
        work[c] = pd.to_numeric(work[c], errors='coerce')
    kept_idx_all, removed_idx_all = [], []
    for tomo, sub in work.groupby(tomo_column, sort=False):
        coords  = sub[[X, Y, Z]].to_numpy(float)
        indices = sub.index.to_numpy()
        scores  = _first_available_score(sub, score_cols)
        ghost_pref = pd.to_numeric(sub.get(ghost_col, 0), errors='coerce').fillna(0).to_numpy(int) \
                     if ghost_col in sub.columns else np.zeros(len(sub), dtype=int)
        order = np.lexsort((-scores, ghost_pref))
        tree   = cKDTree(coords)
        active = np.ones(len(sub), dtype=bool)
        for idx_local in order:
            if not active[idx_local]: continue
            kept_idx_all.append(indices[idx_local])
            for nb in tree.query_ball_point(coords[idx_local], r=min_dist):
                if active[nb] and indices[nb] != indices[idx_local]:
                    removed_idx_all.append(indices[nb])
                active[nb] = False
    kept_idx    = pd.Index(kept_idx_all).unique()
    removed_idx = pd.Index(removed_idx_all).difference(kept_idx)
    return work.loc[kept_idx].copy(), work.loc[removed_idx].copy()


def drop_ghost_singletons(df):
    """Remove ghost particles that are isolated (RowID < 0 or group size ≤ 1)."""
    work = df.copy()
    if 'rlnIsGhost' not in work.columns or 'RowID' not in work.columns:
        return work
    sizes = work.groupby(['_rlnTomoName', 'RowID']).size().rename('grp_size')
    work  = work.join(sizes, on=['_rlnTomoName', 'RowID'])
    mask  = (work['rlnIsGhost'] == 1) & ((work['RowID'] < 0) | (work['grp_size'] <= 1))
    return work.loc[~mask].drop(columns='grp_size')

In [ ]:
# ── Row endpoint extraction and edge filtering ────────────────────────────────

def extract_row_endpoints_mst(df_in, include_ghost_endpoints=False):
    """Extract the two terminal particles of each row using MST leaf detection.
    
    For curved rows the MST is more robust than simple axis projection.
    Returns a DataFrame of endpoint particles with added column 'RowEndpoint'=1.
    """
    X, Y, Z = '_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst'
    TOMO, RID, GHOST = '_rlnTomoName', 'RowID', 'rlnIsGhost'
    work = df_in.copy()
    if GHOST not in work.columns:
        work[GHOST] = 0
    for c in (X, Y, Z):
        work[c] = pd.to_numeric(work[c], errors='coerce')
    out = []
    for (tomo, rid), sub in work.groupby([TOMO, RID], sort=False):
        if rid < 0 or len(sub) == 0: continue
        pts    = sub[[X, Y, Z]].to_numpy(float)
        ghosts = sub[GHOST].to_numpy(int)
        if len(sub) == 1:
            if include_ghost_endpoints or ghosts[0] == 0:
                out.append(sub.iloc[[0]])
            continue
        D      = squareform(pdist(pts))
        csr    = minimum_spanning_tree(D)
        A      = (csr + csr.T).astype(bool)
        deg    = np.ravel(A.sum(axis=1))
        leaves = np.where(deg == 1)[0]
        L = leaves
        if not include_ghost_endpoints:
            real_leaves = leaves[ghosts[leaves] == 0]
            if real_leaves.size >= 2:
                L = real_leaves
        if L.size == 0: continue
        elif L.size == 1:
            out.append(sub.iloc[[int(L[0])]])
        else:
            DD   = squareform(pdist(pts[L]))
            i, j = np.unravel_index(np.argmax(DD), DD.shape)
            out.append(sub.iloc[[int(L[i]), int(L[j])]])
    if not out:
        return work.iloc[[]].copy()
    endpoints = pd.concat(out, ignore_index=False).copy()
    endpoints['RowEndpoint'] = 1
    return endpoints.sort_index()


def endpoint_edge_distance_hist(
    endpoints_df,
    bmask_dir=BMASK_DIR,
    edge_exclude_A=EDGE_EXCLUDE_A,
    tomolist_path=TOMOLIST_PATH,
    mc_level=0.5,
    mc_step=3,
    invert=False,
    use_trimesh=True
):
    """Compute distance of each row endpoint to the lamella boundary surface.
    
    The lamella boundary is defined by a binary mask MRC volume via marching cubes.
    Endpoints closer than edge_exclude_A are flagged and returned separately.
    
    Parameters
    ----------
    bmask_dir      : directory containing <tomo_num>.mrc binary mask volumes
    edge_exclude_A : distance threshold in Å; endpoints below this are excluded
    tomolist_path  : path to two-column file mapping tomo_number to tomo_name
    mc_level       : marching-cubes isovalue (default 0.5 for binary masks)
    mc_step        : marching-cubes step size (larger = faster, coarser mesh)
    invert         : if True, invert the mask before meshing
    use_trimesh    : if True, use trimesh for accurate surface distance; else kd-tree
    
    Returns (kept_endpoints, removed_endpoints, all_endpoints_with_distances)
    """
    import os, mrcfile
    from skimage import measure

    def _load_name2num():
        if 'tomo_mapping' in globals() and isinstance(globals()['tomo_mapping'], dict):
            num2name = globals()['tomo_mapping']
            return {str(v): str(k) for k, v in num2name.items()}
        if tomolist_path is None:
            raise ValueError("Provide TOMOLIST_PATH or define global dict 'tomo_mapping'.")
        name2num = {}
        with open(tomolist_path) as f:
            for line in f:
                s = line.strip()
                if not s or s.startswith('#'): continue
                parts = s.split()
                if len(parts) >= 2:
                    name2num[str(parts[1])] = str(parts[0])
        return name2num

    name2num = _load_name2num()
    X, Y, Z  = '_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst'
    ep = endpoints_df.copy()
    for c in (X, Y, Z):
        ep[c] = pd.to_numeric(ep[c], errors='coerce')

    tri_ok = False
    if use_trimesh:
        try:
            import trimesh; tri_ok = True
        except Exception:
            pass

    recs = []
    for tomo_name, sub in ep.groupby('_rlnTomoName', sort=False):
        tomo_num = name2num.get(str(tomo_name))
        if tomo_num is None:
            print(f"[warn] no mapping for {tomo_name}; skipping {len(sub)} endpoints")
            continue
        mrc_path = os.path.join(bmask_dir, f"{tomo_num}.mrc")
        if not os.path.exists(mrc_path):
            print(f"[warn] missing bmask: {mrc_path}")
            continue
        with mrcfile.open(mrc_path, permissive=True) as m:
            vol = np.asarray(m.data, np.float32)
            if invert: vol = 1.0 - vol
            vz, vy, vx = float(m.voxel_size.z), float(m.voxel_size.y), float(m.voxel_size.x)
            try:
                originA = np.array([float(m.header.origin.x), float(m.header.origin.y),
                                    float(m.header.origin.z)], float)
                if not np.isfinite(originA).all(): originA = np.zeros(3, float)
            except Exception:
                originA = np.zeros(3, float)
            shape_xyz = np.array(vol.shape, int)[::-1]
            center_A  = originA + 0.5 * np.array([vx, vy, vz]) * shape_xyz

        verts_zyx_A, faces, _, _ = measure.marching_cubes(
            vol, level=mc_level, spacing=(vz, vy, vx),
            step_size=int(mc_step), allow_degenerate=False)
        verts_xyz_A = verts_zyx_A[:, [2, 1, 0]]
        if np.any(originA != 0):
            verts_xyz_A += originA[None, :]

        pts_A = sub[[X, Y, Z]].to_numpy(float) + center_A[None, :]

        if tri_ok:
            import trimesh
            mesh = trimesh.Trimesh(vertices=verts_xyz_A, faces=faces.astype(np.int64), process=False)
            _, d, _ = trimesh.proximity.closest_point(mesh, pts_A)
        else:
            tree = cKDTree(verts_xyz_A)
            d, _ = tree.query(pts_A, k=1, workers=-1)
        d = d.astype(np.float32)

        for idx, di in zip(sub.index.to_numpy(), d):
            recs.append({'index': int(idx), '_rlnTomoName': tomo_name, 'dist_to_edge_A': float(di)})
        del vol, verts_zyx_A, faces, verts_xyz_A, pts_A, d

    if not recs:
        raise RuntimeError("No distances computed. Check TOMOLIST_PATH and BMASK_DIR.")

    dist_df = pd.DataFrame.from_records(recs).set_index('index').sort_index()
    out     = endpoints_df.join(dist_df[['dist_to_edge_A']])
    keep_mask = out['dist_to_edge_A'] >= float(edge_exclude_A)

    vals   = out['dist_to_edge_A'].to_numpy(np.float32)
    finite = np.isfinite(vals)
    vmax   = float(np.percentile(vals[finite], 99.5)) if finite.any() else 1.0
    plt.figure()
    plt.hist(vals[finite], bins=np.linspace(0.0, vmax, 50))
    plt.axvline(edge_exclude_A, color='r', linestyle='--', label=f'Exclusion threshold ({edge_exclude_A} Å)')
    plt.xlabel('Distance to lamella boundary (Å)')
    plt.ylabel('Endpoints')
    plt.title('Row endpoint distance to boundary')
    plt.legend()
    plt.tight_layout()
    plt.show()

    return out.loc[keep_mask].copy(), out.loc[~keep_mask].copy(), out

## 4. Load Data

In [ ]:
df, all_lines, dp_idx, loop_idx, data_start, orig_cols = read_data_particles_star(INPUT_STAR)

# Convert coordinate and angle columns to numeric
for c in ['_rlnCenteredCoordinateXAngst', '_rlnCenteredCoordinateYAngst', '_rlnCenteredCoordinateZAngst',
          '_rlnAngleRot', '_rlnAngleTilt', '_rlnAnglePsi']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['RowID'] = -1
print(f"Loaded {len(df)} particles from {df['_rlnTomoName'].nunique()} tomograms.")

## 5. Assign Row Identities

Particles are connected into rows via a k-NN graph where edges require both spatial proximity (< `SPATIAL_TH`) and angular similarity (< `ANGLE_TH_*`) in all three Euler angles. Connected components with ≥ 2 particles are assigned a unique `RowID`; isolated particles remain at `RowID = -1` (singletons).

In [ ]:
global_id = 0

for tomo in df['_rlnTomoName'].unique():
    sub    = df[df['_rlnTomoName'] == tomo]
    coords = sub[['_rlnCenteredCoordinateXAngst',
                  '_rlnCenteredCoordinateYAngst',
                  '_rlnCenteredCoordinateZAngst']].to_numpy()
    eulers = sub[['_rlnAngleRot', '_rlnAngleTilt', '_rlnAnglePsi']].to_numpy()
    idxs   = sub.index.to_numpy()

    nbrs     = NearestNeighbors(n_neighbors=K_NEIGHBORS).fit(coords)
    _, neigh = nbrs.kneighbors(coords)

    G = nx.Graph()
    G.add_nodes_from(idxs)
    for i, neighbors in enumerate(neigh):
        for j in neighbors[1:]:
            if np.linalg.norm(coords[i] - coords[j]) > SPATIAL_TH:
                continue
            if passes_angle_threshold(eulers[i], eulers[j]):
                G.add_edge(idxs[i], idxs[j])

    for comp in nx.connected_components(G):
        if len(comp) > 1:
            for ii in comp:
                df.at[ii, 'RowID'] = global_id
            global_id += 1

n_singletons = (df['RowID'] == -1).sum()
print(f"Assigned {global_id} rows. {n_singletons} singleton particles (RowID = -1).")

## 6. Gap Closing and Deduplication

One round of dilation (ghost particles extrapolated from row endpoints) → reclassification → erosion (remove ghost endpoints) is applied to merge row fragments separated by missing picks. Residual duplicates introduced by dilation are removed, preferring original particles over ghosts.

In [ ]:
closed_df, per_tomo_stats = close_row_gaps(df)
display(per_tomo_stats)
df = closed_df

# Remove ghost singletons introduced by dilation that were not absorbed into rows
df = drop_ghost_singletons(df)

# Remove spatial duplicates, preferring non-ghost / higher-score particles
df, removed_df = clean_with_kdtree_prefer_real(
    df, min_dist=DUP_TH, tomo_column='_rlnTomoName',
    score_cols=['_rlnLCCmax', '_rlnLogLikeliContribution']
)

print(f"After cleaning: {len(df)} particles retained, {len(removed_df)} removed.")

## 7. Extract and Save Singletons

In [ ]:
singletons = df[df['RowID'] == -1].copy()

# Assign random halfsets for singletons (cannot use row-level assignment)
rng = np.random.default_rng(42)
singletons['_rlnRandomSubset'] = rng.integers(1, 3, size=len(singletons)).astype(str)

print(f"{len(singletons)} singletons saved to {SINGLETONS_STAR}")

write_augmented_star(
    df=singletons, lines=all_lines, dp_idx=dp_idx, loop_idx=loop_idx,
    data_start=data_start, orig_cols=orig_cols, output_path=SINGLETONS_STAR
)

## 8. Assign Halfsets and Write Main Output

In [ ]:
# Assign halfsets at row level (entire rows go to one half to avoid correlated signal)
df = assign_halfsets_from_rowid(df, rowid_col='RowID')

write_augmented_star(
    df=df, lines=all_lines, dp_idx=dp_idx, loop_idx=loop_idx,
    data_start=data_start, orig_cols=orig_cols, output_path=OUTPUT_STAR
)

print(f"Wrote {len(df)} particles to {OUTPUT_STAR}")

## 9. Extract Row Endpoints and Filter by Lamella Edge

Row endpoints are identified as MST leaf nodes. Endpoints within `EDGE_EXCLUDE_A` Å of the lamella boundary (as defined by the binary mask) are removed to reduce artefacts from incomplete rows at the lamella edge.

In [ ]:
endpoints_df = extract_row_endpoints_mst(df, include_ghost_endpoints=True)

kept_endpoints, edge_endpoints, endpoints_with_dist = endpoint_edge_distance_hist(
    endpoints_df,
    bmask_dir=BMASK_DIR,
    edge_exclude_A=EDGE_EXCLUDE_A,
    tomolist_path=TOMOLIST_PATH,
    mc_level=0.5,
    mc_step=3,
    invert=False,
    use_trimesh=True
)

print(f"Kept {len(kept_endpoints)} endpoints; removed {len(edge_endpoints)} near lamella edge.")

write_augmented_star(
    df=kept_endpoints, lines=all_lines, dp_idx=dp_idx, loop_idx=loop_idx,
    data_start=data_start, orig_cols=orig_cols, output_path=ENDPOINTS_STAR
)

print(f"Wrote endpoints to {ENDPOINTS_STAR}")

## 10. QC: Per-Tomogram Row Visualisation (PDF)

In [ ]:
with PdfPages(QC_PDF) as pdf:
    for tomo in df['_rlnTomoName'].unique():
        sub = df[df['_rlnTomoName'] == tomo].copy()

        grouped   = sub[sub['RowID'] >= 0]
        singleton = sub[sub['RowID'] < 0]

        fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait

        if len(grouped):
            sc = ax.scatter(
                grouped['_rlnCenteredCoordinateXAngst'],
                grouped['_rlnCenteredCoordinateYAngst'],
                c=grouped['RowID'], cmap='prism', s=6, alpha=0.8, label='Rows'
            )
            cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('Row ID')

        if len(singleton):
            ax.scatter(
                singleton['_rlnCenteredCoordinateXAngst'],
                singleton['_rlnCenteredCoordinateYAngst'],
                c='grey', s=6, alpha=0.8, label='Singletons'
            )

        ax.set_title(f'{tomo}  ({grouped["RowID"].nunique()} rows, {len(singleton)} singletons)')
        ax.set_xlabel('X (Å)')
        ax.set_ylabel('Y (Å)')
        ax.set_aspect('equal')
        ax.legend(markerscale=2, loc='upper right')
        pdf.savefig(fig)
        plt.close(fig)

print(f"QC PDF saved to {QC_PDF}")